## 1. Fetch data from git

### 1.1. Load the data exactly

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

### 1.2. Check dic size

In [3]:
len(documents) 

72

### 1.3. Importar bibliotecas

In [12]:
import os
import json

import numpy as np
import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from tqdm.auto import tqdm

from evaluation_utils import llm_structured

load_dotenv()
openai_client = OpenAI()


## 2. Generating ground truth

### 2.1. Download external components 

In [9]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
!curl -O {PREFIX}/01-agentic-rag/code/rag_helper.py
!curl -O {PREFIX}/04-evaluation/code/evaluation_utils.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2134  100  2134    0     0   5182      0 --:--:-- --:--:-- --:--:--  5179
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3073  100  3073    0     0   2970      0  0:00:01  0:00:01 --:--:--  2971


## Q1: gerando perguntas para as 3 primeiras páginas

In [13]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()


class Questions(BaseModel):
    questions: list[str]

In [14]:
def generate_questions(doc):
    user_prompt = json.dumps({
        "filename": doc["filename"],
        "content": doc["content"],
    })

    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
    )

    return result, usage

In [15]:
first_three = documents[:3]
for doc in first_three:
    print(doc["filename"])


01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/03-rag.md


In [16]:
usages = []

for doc in first_three:
    result, usage = generate_questions(doc)
    usages.append(usage)
    print(doc["filename"], "->", len(result.questions), "perguntas")


01-agentic-rag/lessons/01-intro.md -> 5 perguntas
01-agentic-rag/lessons/02-environment.md -> 5 perguntas
01-agentic-rag/lessons/03-rag.md -> 5 perguntas


In [17]:
# Q1: media de input tokens nas 3 chamadas
input_tokens = [u.input_tokens for u in usages]
avg_input_tokens = sum(input_tokens) / len(input_tokens)

print("input_tokens por chamada:", input_tokens)
print("Média de input tokens:", avg_input_tokens)

options_q1 = [140, 1400, 14000, 140000]
closest_q1 = min(options_q1, key=lambda o: abs(o - avg_input_tokens))
print("Resposta Q1 (mais próxima):", closest_q1)

input_tokens por chamada: [1021, 1287, 1754]
Média de input tokens: 1354.0
Resposta Q1 (mais próxima): 1400


## Q2 e Q3: primeiro resultado de cada busca

In [18]:
!curl -sO {PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv

In [19]:
df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

len(ground_truth), ground_truth[0]

(360,
 {'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'})

In [20]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)


295

In [22]:
from minsearch import Index, VectorSearch

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
text_index.fit(chunks)


def text_search(query, num_results=5):
    return text_index.search(
        query,
        num_results=num_results,
    )


In [23]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [c["content"] for c in chunks]
X = embedding_model.encode(chunk_texts, show_progress_bar=True)

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)


def vector_search(query, num_results=5):
    query_vector = embedding_model.encode(query)
    return vector_index.search(
        query_vector,
        num_results=num_results,
    )


Batches: 100%|██████████| 10/10 [00:00<00:00, 11.05it/s]


In [24]:
q = ground_truth[0]["question"]
print(q)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


In [25]:
# Q2: primeiro resultado do text_search
text_results = text_search(q)
print("Primeiro resultado (text_search):", text_results[0]["filename"])

Primeiro resultado (text_search): 01-agentic-rag/lessons/03-rag.md


In [26]:
# Q3: primeiro resultado do vector_search
vector_results = vector_search(q)
print("Primeiro resultado (vector_search):", vector_results[0]["filename"])

Primeiro resultado (vector_search): 01-agentic-rag/lessons/01-intro.md


## Q4 e Q5 — Métricas de avaliação

In [27]:
def compute_relevance(rec, search_function):
    filename = rec["filename"]
    results = search_function(rec["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance


def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    for rec in tqdm(ground_truth):
        relevance_total.append(compute_relevance(rec, search_function))
    return relevance_total


def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt += 1
    return cnt / len(relevance)


def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance)


def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }


In [28]:
# Q4: Hit Rate do text_search
text_eval = evaluate(ground_truth, text_search)
print("text_search:", text_eval)

options_q4 = [0.55, 0.66, 0.76, 0.88]
closest_q4 = min(options_q4, key=lambda o: abs(o - text_eval["hit_rate"]))
print("Resposta Q4 (mais próxima):", closest_q4)


100%|██████████| 360/360 [00:00<00:00, 2626.35it/s]

text_search: {'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}
Resposta Q4 (mais próxima): 0.76


In [29]:
# Q5: MRR do vector_search
vector_eval = evaluate(ground_truth, vector_search)
print("vector_search:", vector_eval)

options_q5 = [0.35, 0.45, 0.55, 0.65]
closest_q5 = min(options_q5, key=lambda o: abs(o - vector_eval["mrr"]))
print("Resposta Q5 (mais próxima):", closest_q5)

100%|██████████| 360/360 [00:01<00:00, 202.55it/s]

vector_search: {'hit_rate': 0.8083333333333333, 'mrr': 0.6356944444444446}
Resposta Q5 (mais próxima): 0.65


## Q6: tuning do `k` no hybrid search

In [30]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [31]:
k_values = [1, 50, 100, 200]
hybrid_results = []

for k in k_values:
    result = evaluate(
        ground_truth,
        lambda query, k=k: hybrid_search(query, k=k),
    )
    hybrid_results.append({"k": k, **result})
    print(f"k={k}: {result}")

df_hybrid = pd.DataFrame(hybrid_results)
df_hybrid

100%|██████████| 360/360 [00:02<00:00, 166.64it/s]


k=1: {'hit_rate': 0.8583333333333333, 'mrr': 0.6722685185185188}


100%|██████████| 360/360 [00:01<00:00, 207.54it/s]


k=50: {'hit_rate': 0.8472222222222222, 'mrr': 0.6721296296296295}


100%|██████████| 360/360 [00:01<00:00, 234.53it/s]


k=100: {'hit_rate': 0.8472222222222222, 'mrr': 0.6721296296296295}


100%|██████████| 360/360 [00:01<00:00, 233.63it/s]

k=200: {'hit_rate': 0.8472222222222222, 'mrr': 0.6721296296296295}


,k,hit_rate,mrr
0,1,0.858333,0.672269
1,50,0.847222,0.672130
2,100,0.847222,0.672130
3,200,0.847222,0.672130


In [32]:
# Q6: melhor k (em caso de empate, o menor k)
best_mrr = df_hybrid["mrr"].max()
best_k = int(df_hybrid[df_hybrid["mrr"] == best_mrr]["k"].min())

print("Melhor k (Q6):", best_k, "- MRR:", best_mrr)

Melhor k (Q6): 1 - MRR: 0.6722685185185188
